## Fine-Tune Qwen3-8B with LoRA on Ukrainian Literary Style

**Pipeline:**
1. Build SFT splits from `modernized_training_pairs_flat.csv`
2. Prepare MLX data directory
3. Run LoRA fine-tuning with `mlx_lm`
4. Quick test inference

In [11]:
# ── Step 1: Build SFT splits from CSV ────────────────────────────────────────
import json, random, shutil
from pathlib import Path
import pandas as pd

PAIRS_CSV    = "../data/modernized_training_pairs_flat.csv"
SFT_DIR      = Path("../data/sft")
MLX_DIR      = Path("../data/mlx")
ADAPTER_PATH = Path("../models/adapters/qwen3-8b-lora-v2")

SFT_DIR.mkdir(parents=True, exist_ok=True)
MLX_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_PATH.mkdir(parents=True, exist_ok=True)

INSTRUCTION = "Перепиши сучасний український текст у стилі класичної української літератури."

df = pd.read_csv(PAIRS_CSV)
df = (df.dropna(subset=["modern_input", "classic_target"])
        .drop_duplicates(subset=["modern_input", "classic_target"])
        .reset_index(drop=True))

print(f"Total pairs: {len(df)}")
print(f"Models used: {df['model'].value_counts().to_dict()}")

# Shuffle and split 80 / 10 / 10
pairs = df[["modern_input", "classic_target"]].to_dict("records")
random.seed(42)
random.shuffle(pairs)

n       = len(pairs)
n_train = int(n * 0.80)
n_val   = int(n * 0.10)

splits = {
    "train": pairs[:n_train],
    "val":   pairs[n_train : n_train + n_val],
    "test":  pairs[n_train + n_val :],
}

for split_name, rows in splits.items():
    out = SFT_DIR / f"{split_name}.jsonl"
    with open(out, "w", encoding="utf-8") as f:
        for r in rows:
            msg = {
                "messages": [
                    {"role": "user",      "content": f"{INSTRUCTION}\n\nТекст: {r['modern_input']}"},
                    {"role": "assistant", "content": r["classic_target"]},
                ]
            }
            f.write(json.dumps(msg, ensure_ascii=False) + "\n")
    print(f"  {split_name:5s}: {len(rows):4d} examples  →  {out}")

print("\nSFT splits ready.")

Total pairs: 3428
Models used: {'gpt-5.4-mini': 2618, 'gpt-5.5': 774, 'gpt-5.4': 36}
  train: 2742 examples  →  ../data/sft/train.jsonl
  val  :  342 examples  →  ../data/sft/val.jsonl
  test :  344 examples  →  ../data/sft/test.jsonl

SFT splits ready.


## Step 2 — Prepare MLX data directory

MLX-LM expects files named `train.jsonl` and `valid.jsonl` in one folder.

In [12]:
# ── Step 2: Copy SFT files to MLX directory ──────────────────────────────────
# MLX-LM looks for train.jsonl + valid.jsonl (note: NOT val.jsonl)

shutil.copy(SFT_DIR / "train.jsonl", MLX_DIR / "train.jsonl")
shutil.copy(SFT_DIR / "val.jsonl",   MLX_DIR / "valid.jsonl")   # rename!
shutil.copy(SFT_DIR / "test.jsonl",  MLX_DIR / "test.jsonl")

for f in sorted(MLX_DIR.iterdir()):
    lines = sum(1 for _ in open(f))
    print(f"  {f.name}: {lines} examples")

print("\nMLX data dir ready.")

  test.jsonl: 344 examples
  train.jsonl: 2742 examples
  valid.jsonl: 342 examples

MLX data dir ready.


## Step 3 — Run LoRA fine-tuning

**Training notes vs v1:**
- Dataset: ~3400 pairs (was 237) — 14× more data
- Learning rate: `1e-5` (lower to preserve language quality)
- LoRA layers: 16 (was 8) — more expressive adaptation
- Iters: 2000 — ~1.5 epochs over the full training set
- Adapter saved to `models/adapters/qwen3-8b-lora-v2/`

Expected time on M-series: ~90–120 min

In [13]:
# ── Step 3: Fine-tune with MLX-LM ────────────────────────────────────────────
import subprocess, sys, yaml

BASE_MODEL   = "mlx-community/Qwen3-8B-4bit"
ADAPTER_OUT  = str(ADAPTER_PATH.resolve())   # absolute path
DATA_DIR     = str(MLX_DIR.resolve())         # absolute path
CONFIG_PATH  = Path("../mlx_train_v2.yaml").resolve()  # absolute path

# lora_parameters can only be set via YAML config, not CLI flags
config = {
    "model":          BASE_MODEL,
    "data":           DATA_DIR,
    "adapter_path":   ADAPTER_OUT,
    "fine_tune_type": "lora",
    "lora_parameters": {
        "rank":    16,
        "alpha":   32,
        "dropout": 0.05,
        "scale":   2.0,
    },
}

with open(CONFIG_PATH, "w") as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f"Config written → {CONFIG_PATH}")

cmd = [
    sys.executable, "-m", "mlx_lm.lora",
    "-c",                  str(CONFIG_PATH),
    "--train",
    "--num-layers",        "16",
    "--iters",             "2000",
    "--batch-size",        "2",
    "--learning-rate",     "1e-5",
    "--grad-checkpoint",
    "--steps-per-report",  "20",
    "--steps-per-eval",    "200",
    "--val-batches",       "25",
    "--save-every",        "200",
    "--max-seq-length",    "512",
    "--seed",              "42",
]

print("Command:", " ".join(cmd))
print()

result = subprocess.run(cmd)   # no cwd needed — all paths are absolute
if result.returncode != 0:
    print("Training failed — check output above.")
else:
    print(f"\nDone! Adapter saved to {ADAPTER_OUT}")

Config written → /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/final_project/mlx_train_v2.yaml
Command: /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/final_project/.venv/bin/python -m mlx_lm.lora -c /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/final_project/mlx_train_v2.yaml --train --num-layers 16 --iters 2000 --batch-size 2 --learning-rate 1e-5 --grad-checkpoint --steps-per-report 20 --steps-per-eval 200 --val-batches 25 --save-every 200 --max-seq-length 512 --seed 42

Calling `python -m mlx_lm.lora...` directly is deprecated. Use `mlx_lm.lora...` or `python -m mlx_lm lora ...` instead.
Loading configuration file /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/final_project/mlx_train_v2.yaml
Loading pretrained model


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 5542.32it/s]


Loading datasets
Training
Trainable parameters: 0.237% (19.399M/8190.735M)
Starting training..., iters: 2000


Calculating loss...: 100%|██████████| 25/25 [00:45<00:00,  1.81s/it]


Iter 1: Val loss 2.720, Val took 45.342s
[WARNING] Some sequences are longer than 512 tokens. The longest sentence 533 will be truncated to 512. Consider pre-splitting your data to save memory.
Iter 20: Train loss 2.152, Learning Rate 1.000e-05, It/sec 0.162, Tokens/sec 64.077, Trained Tokens 7935, Peak mem 6.335 GB
Iter 40: Train loss 1.500, Learning Rate 1.000e-05, It/sec 0.143, Tokens/sec 62.557, Trained Tokens 16690, Peak mem 6.659 GB
Iter 60: Train loss 1.315, Learning Rate 1.000e-05, It/sec 0.180, Tokens/sec 64.742, Trained Tokens 23890, Peak mem 6.659 GB
[WARNING] Some sequences are longer than 512 tokens. The longest sentence 562 will be truncated to 512. Consider pre-splitting your data to save memory.
[WARNING] Some sequences are longer than 512 tokens. The longest sentence 557 will be truncated to 512. Consider pre-splitting your data to save memory.
Iter 80: Train loss 1.335, Learning Rate 1.000e-05, It/sec 0.118, Tokens/sec 58.964, Trained Tokens 33914, Peak mem 6.659 GB
I

...

Iter 1600: Val loss 1.113, Val took 40.173s
Iter 1600: Train loss 1.025, Learning Rate 1.000e-05, It/sec 0.191, Tokens/sec 64.280, Trained Tokens 637217, Peak mem 6.737 GB
Iter 1600: Saved adapter weights to /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/final_project/models/adapters/qwen3-8b-lora-v2/adapters.safetensors and /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/final_project/models/adapters/qwen3-8b-lora-v2/0001600_adapters.safetensors.
Iter 1620: Train loss 1.069, Learning Rate 1.000e-05, It/sec 0.153, Tokens/sec 65.306, Trained Tokens 645769, Peak mem 6.737 GB
Iter 1640: Train loss 1.058, Learning Rate 1.000e-05, It/sec 0.188, Tokens/sec 65.563, Trained Tokens 652737, Peak mem 6.737 GB
Iter 1660: Train loss 1.039, Learning Rate 1.000e-05, It/sec 0.168, Tokens/sec 66.450, Trained Tokens 660638, Peak mem 6.737 GB
Iter 1680: Train loss 1.047, Learning Rate 1.000e-05, It/sec 0.164, Tokens/sec 61.634, Trained Tokens 668150, Peak mem 6.737 GB
[WARN

Calculating loss...: 100%|██████████| 25/25 [00:40<00:00,  1.63s/it]


Iter 1800: Val loss 1.077, Val took 40.769s
Iter 1800: Train loss 1.122, Learning Rate 1.000e-05, It/sec 0.155, Tokens/sec 62.267, Trained Tokens 714549, Peak mem 6.737 GB
Iter 1800: Saved adapter weights to /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/final_project/models/adapters/qwen3-8b-lora-v2/adapters.safetensors and /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/final_project/models/adapters/qwen3-8b-lora-v2/0001800_adapters.safetensors.
[WARNING] Some sequences are longer than 512 tokens. The longest sentence 540 will be truncated to 512. Consider pre-splitting your data to save memory.
Iter 1820: Train loss 1.129, Learning Rate 1.000e-05, It/sec 0.120, Tokens/sec 57.177, Trained Tokens 724073, Peak mem 6.737 GB
[WARNING] Some sequences are longer than 512 tokens. The longest sentence 545 will be truncated to 512. Consider pre-splitting your data to save memory.
Iter 1840: Train loss 0.948, Learning Rate 1.000e-05, It/sec 0.161, Tokens/sec 61.1

Calculating loss...: 100%|██████████| 25/25 [00:35<00:00,  1.44s/it]


Iter 2000: Val loss 1.098, Val took 36.000s
Iter 2000: Train loss 1.061, Learning Rate 1.000e-05, It/sec 0.176, Tokens/sec 71.693, Trained Tokens 793974, Peak mem 6.737 GB
Iter 2000: Saved adapter weights to /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/final_project/models/adapters/qwen3-8b-lora-v2/adapters.safetensors and /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/final_project/models/adapters/qwen3-8b-lora-v2/0002000_adapters.safetensors.
Saved final weights to /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/final_project/models/adapters/qwen3-8b-lora-v2/adapters.safetensors.

Done! Adapter saved to /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/final_project/models/adapters/qwen3-8b-lora-v2


## Step 4 — Quick test inference

Run a few prompts to sanity-check the fine-tuned adapter before wiring it to the API.

In [15]:
# ── Step 4: Test inference ────────────────────────────────────────────────────
import re
from mlx_lm import load, generate as mlx_generate

if not (ADAPTER_PATH / "adapters.safetensors").exists():
    print(f"Adapter not found at {ADAPTER_PATH} — run Step 3 first.")
else:
    model, tokenizer = load(BASE_MODEL, adapter_path=ADAPTER_OUT)

    SYSTEM = (
        "Ти перетворюєш сучасні українські скарги на текст у стилі класичної "
        "української літератури. Повертай лише перетворений текст. Не пояснюй. "
        "Не пиши російською."
    )

    TEST_INPUTS = [
        "У мене був жахливий день на роботі, я страшенно втомилась.",
        "Він мене кинув і навіть не пояснив чому.",
        "Я одна, і ніхто мене не розуміє.",
        "Грошей немає, робота дістала, все погано.",
        "Я так довго чекала, а він навіть не написав.",
    ]

    for text in TEST_INPUTS:
        messages = [
            {"role": "system", "content": SYSTEM},
            {"role": "user",   "content": f"{INSTRUCTION}\n\nТекст: {text}"},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        raw    = mlx_generate(model, tokenizer, prompt=prompt, max_tokens=300, verbose=False)
        output = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()

        print(f"INPUT : {text}")
        print(f"OUTPUT: {output}")
        print()

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 166293.99it/s]


INPUT : У мене був жахливий день на роботі, я страшенно втомилась.
OUTPUT: Тяжка була робота, тяжка, і я втомилася.

INPUT : Він мене кинув і навіть не пояснив чому.
OUTPUT: Він мене покинув і не сказав, чому.

INPUT : Я одна, і ніхто мене не розуміє.
OUTPUT: Одна, і ніхто не знає, що я така.

INPUT : Грошей немає, робота дістала, все погано.
OUTPUT: Немає грошей, робота дістала, все погано.

INPUT : Я так довго чекала, а він навіть не написав.
OUTPUT: Як довго я чекала, а він не написав.



## Step 5 — Point the API to v2

Once happy with the test outputs, update `.env`:

```
LORA_ADAPTER_PATH=models/adapters/qwen3-8b-lora-v2
MODEL_VERSION=qwen3-8b-lora-v2
INFERENCE_MODE=local
```

Then restart uvicorn.